In [1]:
from pathlib import Path

import pandas as pd

!noglob scp -r hkqai:~/workspace/cc2cc_test5/validate/*.csv ~/workspace/2025.1/validate
name_mol_list = [
    "molecule_W4_11",
    "molecule_G21EA",
    "molecule_G21IP",
    "molecule_DIPCS10",
    "molecule_PA26",
    "molecule_SIE4x4",
    "molecule_ALKBDE10",
    "molecule_YBDE18",
    "molecule_AL2X6",
    "molecule_HEAVYSB11",
    "molecule_NBPRC",
    "molecule_ALK8",
    "molecule_RC21",
    "molecule_G2RC",
    "molecule_BH76",
    "molecule_FH51",
    "molecule_TAUT15",
    "molecule_DC13",
    "molecule_MB16_43",
    "molecule_DARC",
    "molecule_RSE43",
    "molecule_BSR36",
    "molecule_CDIE20",
    "molecule_ISO34",
    "molecule_ISOL24",
    "molecule_C60ISO",
    "molecule_PArel",
    "molecule_BHPERI",
    "molecule_BHDIV10",
    "molecule_INV24",
    "molecule_BHROT27",
    "molecule_PX13",
    "molecule_WCPT18",
    "molecule_RG18",
    "molecule_ADIM6",
    "molecule_S22",
    "molecule_S66",
    "molecule_HEAVY28",
    "molecule_WATER27",
    "molecule_CARBHB12",
    "molecule_PNICO23",
    "molecule_HAL59",
    "molecule_AHB21",
    "molecule_CHB6",
    "molecule_IL16",
    "molecule_IDISP",
    "molecule_ICONF",
    "molecule_ACONF",
    "molecule_Amino20x4",
    "molecule_PCONF21",
    "molecule_MCONF",
    "molecule_SCONF",
    "molecule_UPU23",
    "molecule_BUT14DIOL",
]

if_start_file = {}

for i, name_mol in enumerate(name_mol_list):
    data_path_list = sorted(
        list(Path("../validate").glob(f"*{name_mol}.csv")),
        key=lambda p: p.stat().st_ctime,
    )
    
    for data_path in data_path_list:
        summary_data_path = data_path.stem.split("_" + name_mol)[0] + ".csv"

        with open(data_path, "r") as f:
            data = pd.read_csv(f)

        if if_start_file.get(summary_data_path, True):
            with open(Path("../validate") / summary_data_path, "w") as f2:
                data.to_csv(f2, index=False)
            if_start_file[summary_data_path] = False
        else:
            with open(Path("../validate") / summary_data_path, "a") as f2:
                data.to_csv(f2, index=False, header=False)

for i, name_mol in enumerate(name_mol_list):
    for data_path in list(Path("../validate").glob(f"*{name_mol}.csv")):
        data_path.unlink()

for data_path in list(Path("../validate").glob(f"*test*.csv")):
    data_path.unlink()

ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 8370    97.8KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 5700    69.2KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   27KB 389.4KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 5239    62.6KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 7925    93.2KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 8423   112.6KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   42KB 520.0KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   38KB 291.6KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100% 8947   120.1KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   27KB 366.9KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   18KB 242.1KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pVDZ_mo 100%   17KB 190.0KB/s   00:00    
ccdft_cc-pVDZ_atom-1-1424849_gmtkn-cc-pV

In [2]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

from cc2cc.utils.env_var import DATA_TEST_PATH, DATA_TEST_NO_GRAD_PATH

verbose = 1
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

for data_path in data_path_list:
    data = pd.read_csv(data_path)
    for name in data["name"]:
        if not (DATA_TEST_NO_GRAD_PATH / f"{name}_cc.npz").exists():
            if verbose:
                print(f"Skipping {name} as data file does not exist.")
            continue
        # load the data
        data_frame = np.load(DATA_TEST_NO_GRAD_PATH / f"{name}_cc.npz")
        if "e_dft_d3bj" not in data_frame:
            if verbose:
                print(f"Skipping {name} as e_dft_d3bj is missing.")
            data.loc[data["name"] == name, "delta_d3bj"] = 0
        else:
            data.loc[data["name"] == name, "delta_d3bj"] = (
                data_frame["e_dft_d3bj"].item() - data_frame["e_dft"].item()
            )
        if "e_dft_d3zero" not in data_frame:
            if verbose:
                print(f"Skipping {name} as e_dft_d3zero is missing.")
            data.loc[data["name"] == name, "delta_d3zero"] = 0
        else:
            data.loc[data["name"] == name, "delta_d3zero"] = (
                data_frame["e_dft_d3zero"].item() - data_frame["e_dft"].item()
            )
    # save the processed data
    data.to_csv(data_path, index=False)

In [3]:
import shutil